In [5]:
import math
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.stats import truncnorm

np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [2]:
# Parametres fixes (Section 5.1, p.27)
T = 1.0
dt = 1.0 / 252
n_steps = int(round(T / dt))  # 252
x0 = 1.0
z = 1.4
r_rate = 0.02

# Gardes-fous numeriques
PHI1_CLIP = (-20.0, 20.0)
PHI2_CLIP = (1e-12, 10.0)
MEAN_CLIP = 1e4
LOG_VAR_CLIP = (-50.0, 50.0)
X_DIFF_CLIP = 1e6

print(f"n_steps={n_steps}, dt={dt:.6f}")

n_steps=252, dt=0.003968


In [3]:
def compute_w_star(rho, T=1.0, x0=1.0, z=1.4):
    e = np.exp(rho**2 * T)
    if abs(e - 1) < 1e-12: return z
    return (z * e - x0) / (e - 1)

def simulate_gbm_prices(mu, sigma, n_prices, dt, S0, rng):
    # Simule n_prices points de GBM
    shocks = rng.normal(0, math.sqrt(dt), size=n_prices - 1)
    log_ret = (mu - 0.5 * sigma**2) * dt + sigma * shocks
    S = np.empty(n_prices)
    S[0] = S0
    S[1:] = S0 * np.exp(np.cumsum(log_ret))
    return S

def discounted_wealth_step(x_t, u_t, s_t, s_tp1, r, dt):
    # Richesse actualisee : x_{t+1} = x_t + u_t * (exp(-r*dt)*S_{t+1}/S_t - 1)
    discounted_return = (s_tp1 * math.exp(-r * dt) / s_t) - 1.0
    return x_t + u_t * discounted_return

In [6]:
def V_theta_fn(theta0, theta1, theta2, theta3, w, T, x, t):
    # eq. 44 avec clipping
    x_arr = np.asarray(x, dtype=float)
    t_arr = np.asarray(t, dtype=float)
    xd = np.clip(x_arr - w, -X_DIFF_CLIP, X_DIFF_CLIP)
    return (xd**2) * np.exp(-theta3*(T - t_arr)) + theta2*(t_arr**2) + theta1*t_arr + theta0

def emv_td_error(theta0, theta1, theta2, theta3, w, T, dt, lam, phi1, phi2,
                 vec_xi, vec_xi1, vec_ti, vec_ti1):
    # eq. 42
    vdot = (V_theta_fn(theta0,theta1,theta2,theta3,w,T,vec_xi1,vec_ti1)
          - V_theta_fn(theta0,theta1,theta2,theta3,w,T,vec_xi, vec_ti)) / dt
    return vdot - lam * (phi1 + phi2*(T - vec_ti))

def grad_theta_fn(theta0,theta1,theta2,theta3,w,T,dt,lam,phi1,phi2,
                  vec_xi,vec_xi1,vec_ti,vec_ti1):
    # eq. 47, 48
    td = emv_td_error(theta0,theta1,theta2,theta3,w,T,dt,lam,phi1,phi2,
                      vec_xi,vec_xi1,vec_ti,vec_ti1)
    return float(np.sum(td * dt)), float(np.sum(td * (vec_ti1**2 - vec_ti**2)))

def grad_phi_fn(theta0,theta1,theta2,theta3,w,T,dt,lam,phi1,phi2,
                vec_xi,vec_xi1,vec_ti,vec_ti1):
    # eq. 49, 50 avec clipping
    td = emv_td_error(theta0,theta1,theta2,theta3,w,T,dt,lam,phi1,phi2,
                      vec_xi,vec_xi1,vec_ti,vec_ti1)
    x1 = np.clip(vec_xi1 - w, -X_DIFF_CLIP, X_DIFF_CLIP)
    x0 = np.clip(vec_xi  - w, -X_DIFF_CLIP, X_DIFF_CLIP)
    f1 = 2*(x1**2)*np.exp(-2*phi2*(T-vec_ti1))*(T-vec_ti1)
    f2 = 2*(x0**2)*np.exp(-2*phi2*(T-vec_ti ))*(T-vec_ti )
    sf = -(f1-f2)/dt - lam*(T-vec_ti)
    gp1 = -lam * float(np.sum(td * dt))
    gp2 = float(np.sum(td * dt * sf))
    return gp1, gp2

def emv_policy_mean_var(phi1, phi2, x, w, t, T, lam, rho_sign):
    # eq. 46, en log-space pour stabilite
    sp1 = float(np.clip(phi1, *PHI1_CLIP))
    sp2 = float(np.clip(phi2, *PHI2_CLIP))
    
    # Moyenne (log pour eviter overflow)
    log_coeff = 0.5*math.log(2*sp2/(lam*math.pi)) + 0.5*(2*sp1 - 1)
    log_coeff = float(np.clip(log_coeff, -50, 50))
    mean_coeff = -rho_sign * math.exp(log_coeff)
    mean = float(np.clip(mean_coeff * (x - w), -MEAN_CLIP, MEAN_CLIP))
    
    # Variance (log pour eviter overflow)
    log_var = math.log(1/(2*math.pi)) + 2*sp2*(T-t) + 2*sp1 - 1
    log_var = float(np.clip(log_var, *LOG_VAR_CLIP))
    var = math.exp(log_var)
    
    return mean, var

def sample_truncated_normal(mean_u, var_u, rng=None):
    std_u = math.sqrt(max(var_u, 1e-12))
    a = (0.0 - mean_u) / std_u
    b = np.inf
    return truncnorm.rvs(a, b, loc=mean_u, scale=std_u, random_state=rng)    

print("Fonctions EMV definies.")

Fonctions EMV definies.


In [7]:
def run_emv(mu, sigma, r=0.02, x0=1.0, z=1.4, T=1.0, dt=1.0/252,
            lam=2.0, M=20000, N=10, alpha=0.05, eta_theta=5e-4, eta_phi=5e-4,
            mle_window=100, seed=12345):
    
    nst = int(round(T / dt))
    rho = (mu - r) / sigma
    rho_sign = 1.0 if rho >= 0 else -1.0
    
    # Generation des trajectoires de prix (avec pre-historique de 100 prix pour coherence)
    price_rng = np.random.default_rng(seed)
    action_rng = np.random.default_rng(seed + 999)
    
    # Init (comme dans EMV.ipynb : phi2 = rho^2/2, theta3 = rho^2)
    phi1 = -1.0
    theta1 = 0.0
    theta2 = 1.0
    theta3 = rho**2
    phi2 = theta3 / 2.0
    w = z
    theta0 = -theta2*T**2 - theta1*T - (w-z)**2
    
    policy_phi1 = phi1
    policy_phi2 = phi2
    
    xT_list = []
    w_buffer = []
    hist = {k: [] for k in ['phi1','phi2','theta0','theta1','theta2','theta3','w']}
    
    for k in range(M):
        # Generer les prix de l'episode
        init_price = float(math.exp(price_rng.normal(0, 0.1)))
        prices = simulate_gbm_prices(mu, sigma, mle_window + nst, dt, init_price, price_rng)
        ep_prices = prices[mle_window-1:]  # nst+1 prix pour l'episode
        
        X = [x0]
        times = [0.0]
        
        for i in range(1, nst+1):
            t_prev = times[-1]
            mean_u, var_u = emv_policy_mean_var(
                policy_phi1, policy_phi2, X[-1], w, t_prev, T, lam, rho_sign)
            u_t = sample_truncated_normal(mean_u, var_u, rng=action_rng)
            
            x_next = discounted_wealth_step(X[-1], u_t, ep_prices[i-1], ep_prices[i], r, dt)
            X.append(float(np.clip(x_next, -1e9, 1e9)))
            times.append(float(i * dt))
            
            # Gradients et MAJ a chaque pas (comme EMV.ipynb et le notebook de reference)
            vxi  = np.array(X[:-1], dtype=float)
            vxi1 = np.array(X[1:],  dtype=float)
            vti  = np.array(times[:-1], dtype=float)
            vti1 = np.array(times[1:],  dtype=float)
            
            gt1, gt2 = grad_theta_fn(theta0,theta1,theta2,theta3,w,T,dt,lam,phi1,phi2,
                                     vxi,vxi1,vti,vti1)
            theta1 -= eta_theta * gt1
            theta2 -= eta_theta * gt2
            theta0 = -theta2*T**2 - theta1*T - (w-z)**2
            theta3 = 2*phi2
            
            gp1, gp2 = grad_phi_fn(theta0,theta1,theta2,theta3,w,T,dt,lam,phi1,phi2,
                                   vxi,vxi1,vti,vti1)
            phi1 -= eta_phi * gp1
            phi2 -= eta_phi * gp2
            phi1 = float(np.clip(phi1, *PHI1_CLIP))
            phi2 = float(np.clip(phi2, *PHI2_CLIP))
        
        # Policy improvement
        policy_phi1 = phi1
        policy_phi2 = phi2
        
        x_T = float(X[-1])
        xT_list.append(x_T)
        w_buffer.append(x_T)
        
        # MAJ w (eq. 52)
        if (k+1) % N == 0:
            w -= alpha * (float(np.mean(w_buffer)) - z)
            w_buffer = []
        
        hist['phi1'].append(phi1); hist['phi2'].append(phi2)
        hist['theta0'].append(theta0); hist['theta1'].append(theta1)
        hist['theta2'].append(theta2); hist['theta3'].append(theta3)
        hist['w'].append(w)
    
    return {'xT': np.array(xT_list), **{k: np.array(v) for k,v in hist.items()}}

print("Fonction run_emv definie.")

Fonction run_emv definie.


In [8]:
mu_test, sigma_test = -0.30, 0.10
rho_true = (mu_test - r_rate) / sigma_test

print("EMV en cours...")
t0 = time.time()
emv = run_emv(mu_test, sigma_test, M=20000)
print(f"EMV termine en {time.time()-t0:.0f}s")
last = emv['xT'][-2000:]
me, ve = last.mean(), last.var()
sre = (me-1)/np.sqrt(ve) if ve > 0 else 0
print(f"EMV : E[xT]={me:.4f} Var={ve:.4f} SR={sre:.4f}")
print(f"theta3={emv['theta3'][-1]:.4f} (rho^2={rho_true**2:.4f})")
print(f"w={emv['w'][-1]:.4f} (w*={compute_w_star(rho_true):.4f})")
print()

EMV en cours...
EMV termine en 10264s
EMV : E[xT]=-29676329.5738 Var=138216545702024.5156 SR=-2.5242
theta3=20.0000 (rho^2=10.2400)
w=2482573236.6927 (w*=1.4000)

